# Preparation and Initialization

In [ ]:
# install dependencies
# !pip install -r requirements.txt
!pip install -U ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.2/624.2 kB 10.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.11.0 which is incomp

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/NUS/Classes/CS5242/project

/content/drive/MyDrive/NUS/Classes/CS5242/project


In [ ]:
# %load_ext autoreload
# %autoreload 2

In [3]:
import argparse
import collections
import json
import math
import os
import random
import time
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.manifold import TSNE

from src.data_processing.data_processing import load_mini_imagenet

# Optional FLOPs (handled gracefully if not installed)
try:
    from thop import profile
    THOP_AVAILABLE = True
except Exception:
    THOP_AVAILABLE = False



## Reproducibility & Device

In [4]:
def set_seed(seed=24):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device(use_gpu: bool) -> torch.device:
    if use_gpu and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


## Dataset Utilities


In [5]:
def class_names_from_ds(ds_split):
    return ds_split.features["label"].names


def compute_mean_std(ds_split, num_workers: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute mean/std over a split. Operates over raw images scaled to [0,1].
    Uses a numerically stable running mean/std.
    """
    cnt = 0
    mean = np.zeros(3, dtype=np.float64)
    M2 = np.zeros(3, dtype=np.float64)

    for ex in ds_split:
        img: Image.Image = ex["image"].convert("RGB")
        arr = np.asarray(img, dtype=np.float32) / 255.0  # H, W, C
        arr = arr.reshape(-1, 3)
        batch_n = arr.shape[0]
        batch_mean = arr.mean(axis=0)
        batch_var = arr.var(axis=0)
        # Welford aggregation
        delta = batch_mean - mean
        tot_n = cnt + batch_n
        mean = mean + delta * (batch_n / tot_n)
        M2 = M2 + batch_var * batch_n + (delta**2) * (cnt * batch_n / tot_n)
        cnt = tot_n

    var = M2 / max(cnt, 1)
    std = np.sqrt(var)
    return mean.astype(np.float32), std.astype(np.float32)


def gather_image_meta(ds_split, sample_limit: Optional[int] = None) -> Dict[str, int]:
    """
    Collects resolution and format metadata frequency.
    """
    res_counter = collections.Counter()
    fmt_counter = collections.Counter()
    mode_counter = collections.Counter()
    n = 0
    for ex in ds_split:
        img: Image.Image = ex["image"]
        res_counter[(img.width, img.height)] += 1
        fmt_counter[img.format or "RAW"] += 1
        mode_counter[img.mode] += 1
        n += 1
        if sample_limit and n >= sample_limit:
            break
    return {
        "total": n,
        "resolutions": dict(res_counter),
        "formats": dict(fmt_counter),
        "modes": dict(mode_counter),
    }


def show_random_grid(ds_split, class_names, per_class=5, classes_to_show=6, save_path: Optional[Path] = None):
    """
    Visual inspection: for a subset of classes, show a grid of random samples.
    """
    # Build index per class
    idxs_by_class = collections.defaultdict(list)
    for i, ex in enumerate(ds_split):
        idxs_by_class[ex["label"]].append(i)

    chosen_classes = sorted(random.sample(list(idxs_by_class.keys()), k=min(classes_to_show, len(idxs_by_class))))
    nrows = len(chosen_classes)
    ncols = per_class
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*2.0, nrows*2.0))
    if nrows == 1:
        axes = np.array([axes])
    for r, cls in enumerate(chosen_classes):
        cls_name = class_names[cls]
        picks = random.sample(idxs_by_class[cls], k=min(per_class, len(idxs_by_class[cls])))
        for c, idx in enumerate(picks):
            img = ds_split[idx]["image"]
            axes[r, c].imshow(img)
            if c == 0:
                axes[r, c].set_ylabel(cls_name, fontsize=8)
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
        for c in range(len(picks), ncols):
            axes[r, c].axis('off')
    plt.suptitle("Random Samples by Class")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200)
        print(f"Saved visual inspection grid to: {save_path}")
    plt.close()


def make_transforms(img_size=224, mean=None, std=None):
    """
    Returns (train_transform, eval_transform) using dataset mean/std.
    If mean/std are None, fallback to ImageNet defaults.
    """
    if mean is None or std is None:
        mean = (0.485, 0.456, 0.406)
        std = (0.229, 0.224, 0.225)
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        # transforms.RandomHorizontalFlip(p=0.5),
        # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(img_size * 256 / 224)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    # For "before vs after" visualization, also return a "no-op" tensorizer (no normalization)
    tensor_only = transforms.Compose([
        transforms.Resize(int(img_size * 256 / 224)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor()
    ])
    return train_tf, eval_tf, tensor_only


def visualize_transforms(ds_split, train_tf, eval_tf, tensor_only, save_dir: Path, n=6, seed=0):
    """
    Show original vs eval-transformed and multiple augmented views from train_tf.
    """
    random.seed(seed)
    ensure_dir(save_dir)
    # Pick n random images
    idxs = random.sample(range(len(ds_split)), k=min(n, len(ds_split)))
    for k, i in enumerate(idxs):
        img: Image.Image = ds_split[i]["image"].convert("RGB")

        # Original
        orig = tensor_only(img)  # no normalization, center crop for consistent view
        # Eval transform (normalized) -> de-normalize for display
        eval_t = eval_tf(img)
        eval_disp = denormalize(eval_t, mean=eval_tf.transforms[-1].mean, std=eval_tf.transforms[-1].std)

        # Train transform — show 3 random augmented variants
        aug_imgs = [train_tf(img) for _ in range(3)]
        aug_disps = [denormalize(t, mean=train_tf.transforms[-1].mean, std=train_tf.transforms[-1].std)
                     for t in aug_imgs]

        # Plot
        cols = 2 + len(aug_disps)
        plt.figure(figsize=(3*cols, 3))
        titles = ["Original", "Eval (resized+center crop)"] + [f"Aug {j+1}" for j in range(len(aug_disps))]
        tensors = [orig, eval_disp] + aug_disps
        for j, t in enumerate(tensors):
            plt.subplot(1, cols, j+1)
            plt.imshow(t.permute(1, 2, 0).clamp(0,1).numpy())
            plt.title(titles[j], fontsize=9)
            plt.axis('off')
        out_path = save_dir / f"transforms_{k:02d}.png"
        plt.tight_layout()
        plt.savefig(out_path, dpi=160)
        plt.close()
        print(f"Saved transform visualization: {out_path}")


def denormalize(t: torch.Tensor, mean, std):
    mean = torch.tensor(mean).view(-1,1,1)
    std = torch.tensor(std).view(-1,1,1)
    return (t * std) + mean


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)




# -----------------------------
# Dataloaders
# -----------------------------
class HFDatasetWrapper(torch.utils.data.Dataset):
    """
    Wraps a HF split to return transformed tensors and labels.
    """
    def __init__(self, ds_split, transform):
        self.ds = ds_split
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]
        img: Image.Image = ex["image"].convert("RGB")
        x = self.transform(img)
        y = ex["label"]
        return x, y


def make_loaders(ds, train_tf, eval_tf, batch_size=128, num_workers=2):
    train_ds = HFDatasetWrapper(ds["train"], train_tf)
    val_ds = HFDatasetWrapper(ds["validation"], eval_tf)
    test_ds = HFDatasetWrapper(ds["test"], eval_tf)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                               num_workers=num_workers, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                                             num_workers=num_workers, pin_memory=True)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader


## Models & Feature Extractor

In [6]:
def build_convnext_tiny(num_classes=100, pretrained=True, device=torch.device("cpu")):
    weights = ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
    model = convnext_tiny(weights=weights)
    # Replace classifier head to match 100 classes
    in_features = model.classifier[-1].in_features  # typically 768
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model.to(device)


@torch.no_grad()
def extract_convnext_features(model: nn.Module, loader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extracts GAP features from ConvNeXt-Tiny backbone (before classifier).
    Returns features (N, 768) and labels (N,)
    """
    model.eval()
    feats_list, labels_list = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        # forward_features returns (B, C, H, W)
        feat_map = model.forward_features(x)
        feats = feat_map.mean(dim=(2,3))  # GAP -> (B, C)
        feats_list.append(feats.cpu().numpy())
        labels_list.append(y.numpy())
    feats = np.concatenate(feats_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    return feats, labels


def set_freeze_policy(model: nn.Module, policy: str):
    """
    policy in {'backbone','last_stage','none'}
    """
    # First unfreeze all
    for p in model.parameters():
        p.requires_grad = True

    if policy == "backbone":
        # freeze all except final classifier
        for p in model.features.parameters():
            p.requires_grad = False
    elif policy == "last_stage":
        # freeze all features except last block (heuristic: last 1/4 of features)
        total_blocks = len(list(model.features.children()))
        cutoff = max(0, (3 * total_blocks) // 4)
        for i, m in enumerate(model.features.children()):
            requires = (i >= cutoff)
            for p in m.parameters():
                p.requires_grad = requires
        # classifier remains trainable
    elif policy == "none":
        pass
    else:
        raise ValueError(f"Unknown freeze policy: {policy}")


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def try_flops(model: nn.Module, img_size=224, device=torch.device("cpu")) -> Optional[float]:
    if not THOP_AVAILABLE:
        return None
    dummy = torch.randn(1, 3, img_size, img_size, device=device)
    model_copy = build_convnext_tiny(num_classes=100, pretrained=False, device=device)
    model_copy.load_state_dict(model.state_dict())
    model_copy.eval()
    flops, params = profile(model_copy, inputs=(dummy,), verbose=False)
    # THOP returns FLOPs as number of operations; convert to GFLOPs
    return flops / 1e9


# -----------------------------
# Training / Evaluation
# -----------------------------
def evaluate(model: nn.Module, loader, device: torch.device) -> Tuple[float, float]:
    """
    Returns (top1_acc, avg_loss)
    """
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    ce = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = ce(logits, y)
            loss_sum += loss.item() * y.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / max(total, 1), loss_sum / max(total, 1)


def train_finetune(
    ds, train_tf, eval_tf, device: torch.device,
    epochs=5, batch_size=128, lr=1e-3, freeze_policy="backbone",
    use_pretrained=True, save_dir=Path("./outputs")
):
    ensure_dir(save_dir)
    train_loader, val_loader, test_loader = make_loaders(ds, train_tf, eval_tf, batch_size=batch_size)

    model = build_convnext_tiny(num_classes=100, pretrained=use_pretrained, device=device)
    set_freeze_policy(model, freeze_policy)

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    ce = nn.CrossEntropyLoss()

    # Metrics & logging
    results = {
        "freeze_policy": freeze_policy,
        "use_pretrained": use_pretrained,
        "epochs": epochs,
        "batch_size": batch_size,
        "params_millions": count_params(model) / 1e6,
        "gflops": try_flops(model, device=device),
        "epoch_logs": []
    }

    best_val = 0.0
    ckpt_path = save_dir / f"convnext_{freeze_policy}_{'pre' if use_pretrained else 'scratch'}.pt"

    # GPU memory baseline
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device=device)

    t0_total = time.time()
    for epoch in range(1, epochs+1):
        model.train()
        t_epoch = time.time()
        loss_accum, num_seen = 0.0, 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = ce(logits, y)
            loss.backward()
            optimizer.step()
            loss_accum += loss.item() * y.size(0)
            num_seen += y.size(0)

        train_loss = loss_accum / max(num_seen, 1)
        train_acc, _ = evaluate(model, train_loader, device)
        val_acc, val_loss = evaluate(model, val_loader, device)
        epoch_time = time.time() - t_epoch

        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), ckpt_path)

        log = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "epoch_time_sec": epoch_time
        }
        print(f"[Epoch {epoch:02d}] train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} time={epoch_time:.1f}s")
        results["epoch_logs"].append(log)

    total_time = time.time() - t0_total
    results["total_train_time_sec"] = total_time

    # Inference timing (per image) & peak GPU memory
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    n_imgs, t_inf = 0, 0.0
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(device, non_blocking=True)
            t1 = time.time()
            _ = model(x)
            t2 = time.time()
            t_inf += (t2 - t1)
            n_imgs += x.size(0)
    results["inference_time_per_image_ms"] = (t_inf / max(n_imgs, 1)) * 1000.0
    if device.type == "cuda":
        results["peak_gpu_mem_mb"] = torch.cuda.max_memory_allocated(device=device) / (1024**2)
    else:
        results["peak_gpu_mem_mb"] = None

    # Test accuracy
    test_acc, test_loss = evaluate(model, test_loader, device)
    results["test_acc"] = test_acc
    results["test_loss"] = test_loss

    with open(save_dir / f"results_{freeze_policy}_{'pre' if use_pretrained else 'scratch'}.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Saved training results & checkpoint to:", save_dir)

    return results, ckpt_path


## Classical ML on Frozen Features Function

In [7]:
def classical_ml_experiment(ds, eval_tf, device: torch.device, clf_type="logreg", batch_size=256, save_dir=Path("./outputs")):
    ensure_dir(save_dir)
    # Build feature extractor with pretrained weights; we won't use the classifier head
    model = build_convnext_tiny(num_classes=100, pretrained=True, device=device)
    # Put in eval; we'll only use forward_features + GAP
    train_loader, val_loader, test_loader = make_loaders(ds, eval_tf, eval_tf, batch_size=batch_size)

    print("Extracting features (train/val/test)...")
    t0 = time.time()
    X_train, y_train = extract_convnext_features(model, train_loader, device)
    X_val, y_val = extract_convnext_features(model, val_loader, device)
    X_test, y_test = extract_convnext_features(model, test_loader, device)
    feat_time = time.time() - t0
    print(f"Feature extraction time: {feat_time:.1f}s; Shapes: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}")

    if clf_type == "logreg":
        clf = LogisticRegression(max_iter=2000, n_jobs=-1, verbose=1)
    elif clf_type == "linear_svm":
        clf = LinearSVC()
    else:
        raise ValueError("clf_type must be 'logreg' or 'linear_svm'")

    print(f"Training {clf.__class__.__name__}...")
    t1 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t1

    # Evaluate
    y_val_pred = clf.predict(X_val)
    y_test_pred = clf.predict(X_test)
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    results = {
        "classifier": clf.__class__.__name__,
        "train_time_sec": train_time,
        "feature_time_sec": feat_time,
        "val_acc": float(val_acc),
        "test_acc": float(test_acc),
        "n_train": int(len(y_train)),
        "n_val": int(len(y_val)),
        "n_test": int(len(y_test))
    }
    with open(save_dir / f"classical_ml_{clf_type}.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Classical ML results:", results)
    return results


## Data Exploration Function

In [8]:
def explore(ds, save_dir: Path):
    ensure_dir(save_dir)
    train = ds["train"]
    class_names = class_names_from_ds(train)

    # Class distribution
    counts = collections.Counter([ex["label"] for ex in train])
    dist = [counts.get(i, 0) for i in range(len(class_names))]
    plt.figure(figsize=(12, 4))
    plt.bar(range(len(class_names)), dist)
    plt.title("Class Distribution (Train)")
    plt.xlabel("Class ID")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(save_dir / "class_distribution_train.png", dpi=200)
    plt.close()
    print("Saved class distribution plot.")

    print(f"Number of classes: {len(class_names)}. Balanced? "
          f"min={min(dist)} max={max(dist)} (ideally equal)")

    # Image meta
    meta_train = gather_image_meta(train)
    meta_val = gather_image_meta(ds["validation"])
    meta_test = gather_image_meta(ds["test"])
    with open(save_dir / "image_meta.json", "w") as f:
        json.dump({"train": meta_train, "validation": meta_val, "test": meta_test}, f, indent=2)
    print("Saved image metadata (resolutions/formats/modes).")

    # Visual inspection (random grid)
    show_random_grid(train, class_names, per_class=5, classes_to_show=6, save_path=save_dir / "visual_grid.png")

    # Mean/std for normalization (over train split)
    mean, std = compute_mean_std(train)
    with open(save_dir / "train_mean_std.json", "w") as f:
        json.dump({"mean": mean.tolist(), "std": std.tolist()}, f, indent=2)
    print(f"Computed mean/std: mean={mean}, std={std}. Saved to train_mean_std.json")

## T-SNE visualization Function

In [9]:
def tsne_visualize(ds_split, eval_tf, device: torch.device, save_path: Path, n_samples=2000, seed=42,):
    """
    Extract ConvNeXt GAP features and run 2D t-SNE. Colors by class.
    """
    random.seed(seed)
    idxs = list(range(len(ds_split)))
    if len(idxs) > n_samples:
        idxs = random.sample(idxs, k=n_samples)

    # Build a small loader over the subset
    subset = ds_split.select(idxs)
    loader = torch.utils.data.DataLoader(HFDatasetWrapper(subset, eval_tf), batch_size=256, shuffle=False, num_workers=2)

    device = device
    model = build_convnext_tiny(num_classes=100, pretrained=True, device=device)
    feats, labels = extract_convnext_features(model, loader, device)
    print(f"t-SNE on features shape: {feats.shape}")

    tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=seed, n_iter=1000, verbose=1)
    Z = tsne.fit_transform(feats)

    plt.figure(figsize=(8, 7))
    scatter = plt.scatter(Z[:,0], Z[:,1], c=labels, cmap="tab20", s=6, alpha=0.8)
    plt.title("t-SNE of ConvNeXt-Tiny Features (validation subset)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()
    print(f"Saved t-SNE plot to: {save_path}")

# MAIN

In [11]:
# Define hyper-parameters
USE_GPU = True
RANDOM_SEED=24
SAVE_DIR='experiments/temp'
TASK = {0: 'explore', 1: 'visualize_transforms',
        2: 'features_ml', 3: 'finetune', 4: 'scratch', 5: 'tsne'}[1]
print("TASK: ", TASK)

# training hyper-paramerers
BATCH_SIZE=128
EPOCHES=5
LR=1e-4
IMG_SIZE=224

FREEZE_BLOCK = ["backbone", "last_stage", "none"]

CLF_TYPE=["logreg", "linear_svm"][0]

TSNE_SPLIT = ["train", "validation", "test"][0]

DATA_PATH = "data/"

TASK:  visualize_transforms


## Load dataset:

Just do it once.

In [13]:
set_seed(RANDOM_SEED)
device = get_device(USE_GPU)
save_dir = Path(SAVE_DIR)
ensure_dir(save_dir)

data_dir = Path(DATA_PATH)
ensure_dir(data_dir)

print("Loading dataset: timm/mini-imagenet ...")
ds = load_mini_imagenet(subset=None, cache_path=DATA_PATH)
print({k: len(v) for k, v in ds.items()})


Loading dataset: timm/mini-imagenet ...
{'train': 50000, 'validation': 10000, 'test': 5000}


## EXPORATORY DATA ANALYSIS

In [16]:
ds["train"][0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=357x500>,
 'label': 0}

In [20]:
ds['train'].features

{'image': Image(mode=None, decode=True),
 'label': ClassLabel(names=['n01532829', 'n01558993', 'n01704323', 'n01749939', 'n01770081', 'n01843383', 'n01855672', 'n01910747', 'n01930112', 'n01981276', 'n02074367', 'n02089867', 'n02091244', 'n02091831', 'n02099601', 'n02101006', 'n02105505', 'n02108089', 'n02108551', 'n02108915', 'n02110063', 'n02110341', 'n02111277', 'n02113712', 'n02114548', 'n02116738', 'n02120079', 'n02129165', 'n02138441', 'n02165456', 'n02174001', 'n02219486', 'n02443484', 'n02457408', 'n02606052', 'n02687172', 'n02747177', 'n02795169', 'n02823428', 'n02871525', 'n02950826', 'n02966193', 'n02971356', 'n02981792', 'n03017168', 'n03047690', 'n03062245', 'n03075370', 'n03127925', 'n03146219', 'n03207743', 'n03220513', 'n03272010', 'n03337140', 'n03347037', 'n03400231', 'n03417042', 'n03476684', 'n03527444', 'n03535780', 'n03544143', 'n03584254', 'n03676483', 'n03770439', 'n03773504', 'n03775546', 'n03838899', 'n03854065', 'n03888605', 'n03908618', 'n03924679', 'n039808

In [21]:
class_names = ds['train'].features['label'].names
print(len(class_names))
print(class_names[:10])

100
['n01532829', 'n01558993', 'n01704323', 'n01749939', 'n01770081', 'n01843383', 'n01855672', 'n01910747', 'n01930112', 'n01981276']


In [ ]:
# check class distribution
import pandas as pd
from collections import Counter

labels = dataset['train']['label']
counts = Counter(labels)

df = pd.DataFrame.from_dict(counts, orient='index', columns=['count'])
df = df.sort_index()

print(df.head())

In [14]:

# print("Loading dataset: timm/mini-imagenet ...")
# ds = load_mini_imagenet(subset=None)
# print({k: len(v) for k, v in ds.items()})

# # If exploring, compute mean/std; else try to load from file or fallback to ImageNet stats
# mean_std_path = save_dir / "train_mean_std.json"
# if "explore" in TASK or not mean_std_path.exists():
#     # If explore: we recompute
#     pass
# else:
#     # will attempt to read later
#     pass



# # Run requests
# if "explore" in TASK:
#     explore(ds, save_dir=save_dir)

# # Load mean/std for transforms
# if mean_std_path.exists():
#     with open(mean_std_path, "r") as f:
#         ms = json.load(f)
#         mean, std = np.array(ms["mean"], dtype=np.float32), np.array(ms["std"], dtype=np.float32)
# else:
#     #### WHERE DOES THIS COME FROM? ####


#     # fallback to ImageNet stats if user skipped "explore"
#     mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
#     std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
#     print("Using ImageNet normalization (explore task not run to compute dataset stats).")

# train_tf, eval_tf, tensor_only = make_transforms(img_size=IMG_SIZE, mean=mean, std=std)

# #### I ACTUALLY PREFER TO RUN EACH BLOCK AT A TIME
# if "visualize_transforms" in TASK:
#     vis_dir = save_dir / "transform_viz"
#     visualize_transforms(ds["train"], train_tf, eval_tf, tensor_only, save_dir=vis_dir, n=8, seed=0)

# if "features_ml" in TASK:
#     classical_ml_experiment(ds, eval_tf, device=device, clf_type=CLF_TYPE, batch_size=max(128, BATCH_SIZE), save_dir=save_dir)

# if "finetune" in TASK:
#     # Pretrained fine-tuning with chosen freeze policy
#     train_finetune(ds, train_tf, eval_tf, device=device, epochs=EPOCHES, batch_size=BATCH_SIZE,
#                     lr=LR, freeze_policy=FREEZE_BLOCK, use_pretrained=True, save_dir=save_dir)

# if "scratch" in TASK:
#     # Train from scratch baseline
#     train_finetune(ds, train_tf, eval_tf, device=device, epochs=EPOCHES, batch_size=BATCH_SIZE,
#                     lr=LR, freeze_policy="none", use_pretrained=False, save_dir=save_dir)

# if "tsne" in TASK:
#     split = ds[TSNE_SPLIT]
#     tsne_visualize(split, eval_tf, device=device, save_path=save_dir / f"tsne_{TSNE_SPLIT}.png", n_samples=2000)



In [12]:
if "explore" in TASK:
    explore(ds, save_dir=save_dir)